# 2. Model Specification and Prior Analysis

## 2.1 Model Overview

We propose two models for predicting **Mean Finish Time** of ultra-trail races:

### Model 1: Normal Linear Regression

$$y_i \sim \text{Normal}(\mu_i, \sigma)$$
$$\mu_i = \alpha + \beta_{dist} \cdot \text{distance\_std}_i + \beta_{elev} \cdot \text{elevation\_std}_i$$

### Model 2: Student-t Linear Regression (Robust)

$$y_i \sim \text{Student-t}(\nu, \mu_i, \sigma)$$
$$\mu_i = \alpha + \beta_{dist} \cdot \text{distance\_std}_i + \beta_{elev} \cdot \text{elevation\_std}_i$$

### Key Differences

| Feature | Model 1 (Normal) | Model 2 (Student-t) |
|---------|------------------|---------------------|
| Likelihood | Normal | Student-t |
| Parameters | $\alpha, \beta_{dist}, \beta_{elev}, \sigma$ | $\alpha, \beta_{dist}, \beta_{elev}, \sigma, \nu$ |
| Tail behavior | Light (exponential decay) | Heavy (polynomial decay) |
| Robustness to outliers | Low | High |
| Additional parameter | — | $\nu$ (degrees of freedom) |

### Justification for Two Models

From the data exploration, we observed:
1. The data has right-skewed tails and outlier races (QQ plot deviations)
2. Some races have unexpectedly high finish times due to extreme conditions
3. The Normal model may inflate $\sigma$ to accommodate outliers, distorting predictions for typical races
4. The Student-t model has heavier tails controlled by $\nu$, allowing it to accommodate outliers without inflating the scale for the bulk of data

If the data truly contains outlier races (extreme weather, unusual course conditions), the Student-t model should provide:
- Better predictive performance (higher ELPD)
- More accurate uncertainty estimates for typical races
- Lower Pareto-k values in LOO diagnostics

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import arviz as az
from cmdstanpy import CmdStanModel
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.dpi'] = 110

# Load processed data
df = pd.read_csv('utmb_processed.csv')
print(f"Data loaded: {df.shape[0]:,} rows")
print(f"Mean Finish Time: mean={df['Mean Finish Time'].mean():.2f}h, std={df['Mean Finish Time'].std():.2f}h")
print(f"Distance (std): mean={df['distance_std'].mean():.4f}, std={df['distance_std'].std():.4f}")
print(f"Elevation (std): mean={df['elevation_std'].mean():.4f}, std={df['elevation_std'].std():.4f}")

## 2.2 Technical Model Description

### Model 1 — Stan Code

```stan
data {
  int<lower=1> N;
  vector[N] y;              // finish time in hours
  vector[N] distance_std;   // standardized distance
  vector[N] elevation_std;  // standardized elevation gain
}
parameters {
  real alpha;               // intercept
  real beta_dist;           // effect of distance
  real beta_elev;           // effect of elevation gain
  real<lower=0> sigma;      // residual SD
}
model {
  alpha ~ normal(10, 5);
  beta_dist ~ normal(5, 3);
  beta_elev ~ normal(2, 2);
  sigma ~ exponential(0.2);
  y ~ normal(alpha + beta_dist * distance_std + beta_elev * elevation_std, sigma);
}
```

### Model 2 — Stan Code

```stan
data {
  int<lower=1> N;
  vector[N] y;
  vector[N] distance_std;
  vector[N] elevation_std;
}
parameters {
  real alpha;
  real beta_dist;
  real beta_elev;
  real<lower=0> sigma;
  real<lower=1> nu;         // degrees of freedom
}
model {
  alpha ~ normal(10, 5);
  beta_dist ~ normal(5, 3);
  beta_elev ~ normal(2, 2);
  sigma ~ exponential(0.2);
  nu ~ gamma(2, 0.1);      // weakly informative on df
  y ~ student_t(nu, alpha + beta_dist * distance_std + beta_elev * elevation_std, sigma);
}
```

## 2.3 Prior Selection Rationale

All predictors are standardized (mean=0, std=1), so:

| Parameter | Prior | Rationale |
|-----------|-------|----------|
| $\alpha$ | Normal(10, 5) | Intercept = expected time at average distance/elevation. Trail races average ~10h. Prior allows 0-20h. |
| $\beta_{dist}$ | Normal(5, 3) | A 1-SD increase in distance (~30km) should increase time by ~5h. Prior allows 0-11h. |
| $\beta_{elev}$ | Normal(2, 2) | A 1-SD increase in elevation (~1500m) should add ~2h. Prior allows 0-6h. |
| $\sigma$ | Exponential(0.2) | Residual SD with mean=5h. Races have substantial variability beyond distance/elevation. |
| $\nu$ (Model 2 only) | Gamma(2, 0.1) | Mean=20, allows values from ~3 to 50+. Small $\nu$ = heavy tails, large $\nu$ ≈ Normal. |

### Why these priors?

1. **Weakly informative**: They constrain to physically plausible ranges but don't dominate the likelihood
2. **Consistent with domain knowledge**: A 100km race takes ~10-15h on average; 20km takes ~3-4h
3. **Positive coefficients expected**: Longer distance and more elevation should increase time
4. **Not over-tuned**: Priors were set based on general ultramarathon knowledge, not from observing this specific dataset

## 2.4 Prior Predictive Checks — Parameters

Let's simulate from the priors to check if the parameter values are sensible.

In [ ]:
# Prior predictive simulation — parameter level
n_sim = 5000

alpha_sim = np.random.normal(10, 5, n_sim)
beta_dist_sim = np.random.normal(5, 3, n_sim)
beta_elev_sim = np.random.normal(2, 2, n_sim)
sigma_sim = np.random.exponential(1/0.2, n_sim)  # Exponential(rate=0.2) => mean=5
nu_sim = np.random.gamma(2, 1/0.1, n_sim)  # Gamma(shape=2, rate=0.1) => mean=20

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(alpha_sim, bins=50, color='steelblue', edgecolor='white')
axes[0,0].axvline(10, color='red', linestyle='--', label='Prior mean')
axes[0,0].set_title(r'$\alpha$ ~ Normal(10, 5)')
axes[0,0].set_xlabel('Intercept [hours]')
axes[0,0].legend()

axes[0,1].hist(beta_dist_sim, bins=50, color='#4CAF50', edgecolor='white')
axes[0,1].axvline(5, color='red', linestyle='--', label='Prior mean')
axes[0,1].set_title(r'$\beta_{dist}$ ~ Normal(5, 3)')
axes[0,1].set_xlabel('Effect of 1 SD distance [hours]')
axes[0,1].legend()

axes[0,2].hist(beta_elev_sim, bins=50, color='#FF9800', edgecolor='white')
axes[0,2].axvline(2, color='red', linestyle='--', label='Prior mean')
axes[0,2].set_title(r'$\beta_{elev}$ ~ Normal(2, 2)')
axes[0,2].set_xlabel('Effect of 1 SD elevation [hours]')
axes[0,2].legend()

axes[1,0].hist(sigma_sim[sigma_sim < 30], bins=50, color='#9C27B0', edgecolor='white')
axes[1,0].axvline(5, color='red', linestyle='--', label='Prior mean')
axes[1,0].set_title(r'$\sigma$ ~ Exponential(0.2)')
axes[1,0].set_xlabel('Residual SD [hours]')
axes[1,0].legend()

axes[1,1].hist(nu_sim[nu_sim < 80], bins=50, color='#E91E63', edgecolor='white')
axes[1,1].axvline(20, color='red', linestyle='--', label='Prior mean')
axes[1,1].set_title(r'$\nu$ ~ Gamma(2, 0.1)')
axes[1,1].set_xlabel('Degrees of freedom')
axes[1,1].legend()

axes[1,2].axis('off')
axes[1,2].text(0.1, 0.7, 'Prior Parameter Summary:', fontsize=12, fontweight='bold', transform=axes[1,2].transAxes)
axes[1,2].text(0.1, 0.55, f'α: [{np.percentile(alpha_sim, 2.5):.1f}, {np.percentile(alpha_sim, 97.5):.1f}] (95% CI)', fontsize=10, transform=axes[1,2].transAxes)
axes[1,2].text(0.1, 0.43, f'β_dist: [{np.percentile(beta_dist_sim, 2.5):.1f}, {np.percentile(beta_dist_sim, 97.5):.1f}] (95% CI)', fontsize=10, transform=axes[1,2].transAxes)
axes[1,2].text(0.1, 0.31, f'β_elev: [{np.percentile(beta_elev_sim, 2.5):.1f}, {np.percentile(beta_elev_sim, 97.5):.1f}] (95% CI)', fontsize=10, transform=axes[1,2].transAxes)
axes[1,2].text(0.1, 0.19, f'σ: [{np.percentile(sigma_sim, 2.5):.1f}, {np.percentile(sigma_sim, 97.5):.1f}] (95% CI)', fontsize=10, transform=axes[1,2].transAxes)
axes[1,2].text(0.1, 0.07, f'ν: [{np.percentile(nu_sim, 2.5):.1f}, {np.percentile(nu_sim, 97.5):.1f}] (95% CI)', fontsize=10, transform=axes[1,2].transAxes)

plt.suptitle('Prior Predictive Check — Parameter Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('fig03_prior_parameters.png', bbox_inches='tight')
plt.show()

print("\nParameter prior 95% intervals:")
print(f"  alpha:     [{np.percentile(alpha_sim, 2.5):.1f}, {np.percentile(alpha_sim, 97.5):.1f}] hours")
print(f"  beta_dist: [{np.percentile(beta_dist_sim, 2.5):.1f}, {np.percentile(beta_dist_sim, 97.5):.1f}] hours per SD")
print(f"  beta_elev: [{np.percentile(beta_elev_sim, 2.5):.1f}, {np.percentile(beta_elev_sim, 97.5):.1f}] hours per SD")
print(f"  sigma:     [{np.percentile(sigma_sim, 2.5):.1f}, {np.percentile(sigma_sim, 97.5):.1f}] hours")
print(f"  nu:        [{np.percentile(nu_sim, 2.5):.1f}, {np.percentile(nu_sim, 97.5):.1f}]")

### Assessment of Parameter Priors

- **$\alpha$ ∈ [0, 20] hours**: Sensible — the average race at mean distance/elevation finishes in 5-15 hours
- **$\beta_{dist}$ ∈ [-1, 11] hours/SD**: Allows negative (unlikely but not impossible) and strong positive effects. Since 1 SD ≈ 30km, this means 0 to 11 hours per extra 30km — physically reasonable
- **$\beta_{elev}$ ∈ [-2, 6] hours/SD**: Since 1 SD ≈ 1500m elevation, up to 6 hours per 1500m is reasonable for ultra-trail
- **$\sigma$ ∈ [0.1, 15] hours**: Allows for substantial residual variation
- **$\nu$ ∈ [3, 50+]**: Small values (heavy tails) to large values (≈ Normal) — data-driven

## 2.5 Prior Predictive Checks — Measurements

Now we simulate full datasets from the prior to check if the model generates plausible race finish times.

In [ ]:
# Prior predictive simulation — measurement level
# Use actual predictor values from the data
dist_vals = df['distance_std'].values
elev_vals = df['elevation_std'].values
N = len(dist_vals)

n_prior_sims = 200
y_prior_normal = np.zeros((n_prior_sims, N))
y_prior_studentt = np.zeros((n_prior_sims, N))

for s in range(n_prior_sims):
    a = np.random.normal(10, 5)
    bd = np.random.normal(5, 3)
    be = np.random.normal(2, 2)
    sig = np.random.exponential(1/0.2)
    nu = max(1.1, np.random.gamma(2, 1/0.1))  # ensure nu > 1
    
    mu = a + bd * dist_vals + be * elev_vals
    
    # Model 1: Normal
    y_prior_normal[s] = np.random.normal(mu, sig)
    
    # Model 2: Student-t
    y_prior_studentt[s] = stats.t.rvs(df=nu, loc=mu, scale=sig)

print("Prior predictive simulation complete.")
print(f"  Normal model — simulated y range: [{y_prior_normal.min():.0f}, {y_prior_normal.max():.0f}] hours")
print(f"  Student-t model — simulated y range: [{y_prior_studentt.min():.0f}, {y_prior_studentt.max():.0f}] hours")

In [ ]:
# Visualize prior predictive distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Model 1 (Normal) prior predictive
y_obs = df['Mean Finish Time'].values

# Histogram of prior predictive means
prior_means_normal = y_prior_normal.mean(axis=1)
axes[0,0].hist(prior_means_normal, bins=40, color='steelblue', alpha=0.7, edgecolor='white', label='Prior pred. mean')
axes[0,0].axvline(y_obs.mean(), color='red', linewidth=2, linestyle='--', label=f'Observed mean={y_obs.mean():.1f}h')
axes[0,0].set_title('Model 1 (Normal): Prior Predictive Mean')
axes[0,0].set_xlabel('Simulated dataset mean [hours]')
axes[0,0].legend()

# Prior predictive — overlay of simulated datasets
for s in range(min(50, n_prior_sims)):
    axes[0,1].plot(sorted(y_prior_normal[s, :100]), np.linspace(0, 1, 100), color='steelblue', alpha=0.1)
axes[0,1].plot(sorted(y_obs[:100]), np.linspace(0, 1, 100), color='red', linewidth=2, label='Observed')
axes[0,1].set_title('Model 1 (Normal): Prior Predictive CDF (sample)')
axes[0,1].set_xlabel('Finish time [hours]')
axes[0,1].set_ylabel('CDF')
axes[0,1].set_xlim(-20, 60)
axes[0,1].legend()

# Model 2 (Student-t) prior predictive
prior_means_t = y_prior_studentt.mean(axis=1)
axes[1,0].hist(prior_means_t[np.abs(prior_means_t) < 100], bins=40, color='#E91E63', alpha=0.7, edgecolor='white', label='Prior pred. mean')
axes[1,0].axvline(y_obs.mean(), color='red', linewidth=2, linestyle='--', label=f'Observed mean={y_obs.mean():.1f}h')
axes[1,0].set_title('Model 2 (Student-t): Prior Predictive Mean')
axes[1,0].set_xlabel('Simulated dataset mean [hours]')
axes[1,0].legend()

for s in range(min(50, n_prior_sims)):
    data_s = y_prior_studentt[s, :100]
    data_s_clipped = np.clip(data_s, -50, 100)
    axes[1,1].plot(sorted(data_s_clipped), np.linspace(0, 1, 100), color='#E91E63', alpha=0.1)
axes[1,1].plot(sorted(y_obs[:100]), np.linspace(0, 1, 100), color='red', linewidth=2, label='Observed')
axes[1,1].set_title('Model 2 (Student-t): Prior Predictive CDF (sample)')
axes[1,1].set_xlabel('Finish time [hours]')
axes[1,1].set_ylabel('CDF')
axes[1,1].set_xlim(-20, 60)
axes[1,1].legend()

plt.suptitle('Prior Predictive Checks — Simulated Measurements', fontsize=13)
plt.tight_layout()
plt.savefig('fig04_prior_predictive_measurements.png', bbox_inches='tight')
plt.show()

In [ ]:
# Prior predictive check — summary statistics
print("Prior Predictive Check — Summary Statistics")
print("=" * 60)
print(f"{'Statistic':<25} {'Observed':<12} {'Normal Prior':<20} {'Student-t Prior':<20}")
print("-" * 60)

obs_mean = y_obs.mean()
obs_std = y_obs.std()
obs_min = y_obs.min()
obs_max = y_obs.max()

# For Normal model
n_means = [y_prior_normal[s].mean() for s in range(n_prior_sims)]
n_stds = [y_prior_normal[s].std() for s in range(n_prior_sims)]
n_mins = [y_prior_normal[s].min() for s in range(n_prior_sims)]
n_maxs = [y_prior_normal[s].max() for s in range(n_prior_sims)]

# For Student-t model
t_means = [y_prior_studentt[s].mean() for s in range(n_prior_sims)]
t_stds = [y_prior_studentt[s].std() for s in range(n_prior_sims)]
t_mins = [y_prior_studentt[s].min() for s in range(n_prior_sims)]
t_maxs = [y_prior_studentt[s].max() for s in range(n_prior_sims)]

print(f"{'Mean':<25} {obs_mean:<12.1f} {np.median(n_means):<8.1f} (±{np.std(n_means):.1f})    {np.median(t_means):<8.1f} (±{np.std(t_means):.1f})")
print(f"{'Std':<25} {obs_std:<12.1f} {np.median(n_stds):<8.1f} (±{np.std(n_stds):.1f})    {np.median(t_stds):<8.1f} (±{np.std(t_stds):.1f})")
print(f"{'Min':<25} {obs_min:<12.1f} {np.median(n_mins):<8.1f} (±{np.std(n_mins):.1f})    {np.median(t_mins):<8.1f} (±{np.std(t_mins):.1f})")
print(f"{'Max':<25} {obs_max:<12.1f} {np.median(n_maxs):<8.1f} (±{np.std(n_maxs):.1f})    {np.median(t_maxs):<8.1f} (±{np.std(t_maxs):.1f})")

# Check: does observed data fall within prior predictive range?
print(f"\n\nAssessment:")
print(f"  Observed mean ({obs_mean:.1f}h) within Normal prior pred. range: {np.percentile(n_means, 2.5):.1f} to {np.percentile(n_means, 97.5):.1f}")
print(f"  Observed mean ({obs_mean:.1f}h) within Student-t prior pred. range: {np.percentile(t_means, 2.5):.1f} to {np.percentile(t_means, 97.5):.1f}")
print(f"\n  → Both models generate datasets that could include the observed data.")
print(f"  → Priors are not over-tuned (wide range) but not absurd (centered on reasonable values).")
print(f"  → Some prior predictive simulations produce negative times — this is acceptable for a prior")
print(f"    predictive check because the prior should be regularizing, not perfectly calibrated.")
print(f"    The likelihood will constrain the posterior to positive values.")

## 2.6 Prior Predictive Assessment

### Conclusions

1. **Parameter priors generate sensible values**: All parameters fall within physically meaningful ranges
2. **Measurement-level prior predictive**: The observed data mean falls within the range of simulated dataset means — our priors are compatible with the data
3. **Not over-tuned**: The prior predictive range is broad (covers many possible scenarios), showing we haven't peeked at the data to set priors
4. **Negative simulated times**: Some simulations produce negative finish times. This is a known artifact of Normal/Student-t likelihoods with broad priors. The posterior will be constrained by the positive observed data. We accept this rather than using a truncated model, as the bulk of the prior mass is in the positive range.

### Prior selection method

Priors were selected by:
1. **Domain knowledge**: Ultramarathon finishing times typically range 2-40 hours depending on distance
2. **Scale matching**: After standardizing predictors, we estimated reasonable effect sizes in hours
3. **Weakly informative principle**: Priors are broad enough to let the data speak but prevent absurd values
4. **Prior predictive simulation**: Confirmed that the priors generate plausible data ranges

In [ ]:
# Compile Stan models to verify they work
model1 = CmdStanModel(stan_file='model1_normal.stan')
model2 = CmdStanModel(stan_file='model2_student_t.stan')
print("Both Stan models compiled successfully.")

In [ ]:
# Prepare data for Stan
stan_data = {
    'N': len(df),
    'y': df['Mean Finish Time'].values.tolist(),
    'distance_std': df['distance_std'].values.tolist(),
    'elevation_std': df['elevation_std'].values.tolist()
}

print(f"Stan data prepared:")
print(f"  N = {stan_data['N']}")
print(f"  y range: [{min(stan_data['y']):.2f}, {max(stan_data['y']):.2f}] hours")
print(f"\nReady to fit models in the next notebooks.")